# Export XGBoost Model to ONNX Format

This notebook exports a trained XGBoost model to ONNX (Open Neural Network Exchange) format for deployment in the Go trading bot.

## Why ONNX?

- **Cross-platform**: ONNX models can be loaded in Go, C++, Rust, etc.
- **Optimized inference**: ONNX Runtime provides fast, production-ready inference
- **Standard format**: Industry-standard format supported by many frameworks
- **No Python dependency**: Go bot doesn't need Python to run inference

## Workflow

1. Load trained XGBoost model (.joblib or .json)
2. Convert to ONNX format with proper input/output names
3. Verify the ONNX model is valid
4. Test inference and validate predictions match
5. Generate final report

## 1. Setup and Imports

In [ ]:
import json
from pathlib import Path
import time

import joblib
import numpy as np
import pandas as pd
import onnx
import xgboost as xgb
from onnxconverter_common import FloatTensorType
from onnxmltools.convert import convert_xgboost
import onnxruntime as ort

print("✓ All packages imported successfully")
print(f"XGBoost version: {xgb.__version__}")
print(f"ONNX version: {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

## 2. Configuration

Set paths to your trained model files:

In [ ]:
# Input paths
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "xgboost_model.joblib"  # or "xgboost_model.json"
FEATURES_PATH = MODEL_DIR / "features.json"
METRICS_PATH = MODEL_DIR / "metrics.json"

# Output path
ONNX_OUTPUT_PATH = MODEL_DIR / "xgboost_model.onnx"

print("Configuration:")
print(f"  Model path:    {MODEL_PATH}")
print(f"  Features path: {FEATURES_PATH}")
print(f"  Output path:   {ONNX_OUTPUT_PATH}")
print(f"\nChecking files...")
print(f"  Model exists:    {MODEL_PATH.exists()}")
print(f"  Features exist:  {FEATURES_PATH.exists()}")
print(f"  Metrics exist:   {METRICS_PATH.exists()}")

## 3. Load Model and Feature Information

In [ ]:
print(f"Loading model from {MODEL_PATH}...")

# Load model (supports both .joblib and .json formats)
if MODEL_PATH.suffix == ".joblib":
    model = joblib.load(MODEL_PATH)
    print("✓ Loaded model from joblib format")
else:
    model = xgb.XGBClassifier()
    model.load_model(MODEL_PATH)
    print("✓ Loaded model from XGBoost JSON format")

# Load feature names
with open(FEATURES_PATH) as f:
    features = json.load(f)

n_features = len(features)
print(f"\nModel information:")
print(f"  Number of features: {n_features}")
print(f"  Number of classes: {model.n_classes_}")
print(f"\nFeature names:")
for i, feat in enumerate(features, 1):
    print(f"  {i:2d}. {feat}")

## 4. Load Training Metrics

In [ ]:
if METRICS_PATH.exists():
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    
    print("Training Metrics:")
    print(f"  Train Accuracy:  {metrics.get('train_accuracy', 'N/A')}")
    print(f"  Val Accuracy:    {metrics.get('val_accuracy', 'N/A')}")
    print(f"  Train F1:        {metrics.get('train_f1_weighted', 'N/A')}")
    print(f"  Val F1:          {metrics.get('val_f1_weighted', 'N/A')}")
    print(f"  Train size:      {metrics.get('train_size', 'N/A'):,}")
    print(f"  Val size:        {metrics.get('val_size', 'N/A'):,}")
else:
    print("No metrics file found")

## 5. Test Model with Random Input (Pre-Export)

Verify the model works before converting to ONNX:

In [ ]:
# Generate random test input
test_input = np.random.randn(5, n_features).astype(np.float32)

# Make predictions
predictions = model.predict(test_input)
probabilities = model.predict_proba(test_input)

label_names = {0: 'DOWN', 1: 'NEUTRAL', 2: 'UP'}

print("Test predictions (XGBoost model):")
print(f"Input shape: {test_input.shape}\n")

for i in range(len(test_input)):
    pred_label = predictions[i]
    pred_proba = probabilities[i]
    print(f"Sample {i+1}:")
    print(f"  Predicted: {label_names[pred_label]}")
    print(f"  Probabilities: DOWN={pred_proba[0]:.3f}, NEUTRAL={pred_proba[1]:.3f}, UP={pred_proba[2]:.3f}")
    print()

## 6. Convert to ONNX Format

### Important Notes:
- ONNX conversion requires feature names in format `f0`, `f1`, ..., `fN`
- We map original feature names to safe names for conversion
- Output names are normalized to `label` and `probabilities` for Go runtime

In [ ]:
print("Preparing model for ONNX conversion...\n")

# Create safe feature names required by onnxmltools ('f0', 'f1', ...)
safe_feature_names = [f"f{i}" for i in range(n_features)]
print(f"Mapping {n_features} features to safe names (f0..f{n_features-1})")

# Set feature names on the underlying booster
try:
    # If model is sklearn wrapper
    booster = model.get_booster()
    booster.feature_names = safe_feature_names
    # Also set on sklearn wrapper if attribute exists
    try:
        model.feature_names_in_ = safe_feature_names
    except Exception:
        pass
    print("✓ Set feature names on sklearn wrapper booster")
except Exception:
    try:
        # If model is a raw Booster
        model.feature_names = safe_feature_names
        print("✓ Set feature names on raw booster")
    except Exception as e:
        print(f"⚠ Warning: unable to set booster feature names: {e}")

# Define input type for ONNX conversion
initial_type = [("float_input", FloatTensorType([None, n_features]))]

print("\nConverting to ONNX...")
onnx_model = convert_xgboost(
    model,
    initial_types=initial_type,
    target_opset=12,
)

print("✓ Conversion successful!")

## 7. Normalize Output Names for Go Runtime

In [ ]:
print("Normalizing output names...\n")

# Normalize output names to expected names
for output in onnx_model.graph.output:
    old_name = output.name
    if "probabilities" in output.name.lower() or output.name == "output_probability":
        output.name = "probabilities"
        print(f"  Output: {old_name} → probabilities")
    elif "label" in output.name.lower():
        output.name = "label"
        print(f"  Output: {old_name} → label")

# Also normalize intermediate node outputs
for node in onnx_model.graph.node:
    for i, out in enumerate(node.output):
        if "probabilities" in out.lower() or out == "output_probability":
            node.output[i] = "probabilities"

print("\n✓ Output names normalized for Go runtime")

## 8. Save ONNX Model

In [ ]:
# Create output directory if needed
ONNX_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save model
onnx.save_model(onnx_model, str(ONNX_OUTPUT_PATH))

file_size = ONNX_OUTPUT_PATH.stat().st_size / (1024 * 1024)  # Convert to MB
print(f"✓ Saved ONNX model to {ONNX_OUTPUT_PATH}")
print(f"  File size: {file_size:.2f} MB")

## 9. Verify ONNX Model

In [ ]:
print("Verifying ONNX model...\n")

# Load and check model
onnx_model = onnx.load(str(ONNX_OUTPUT_PATH))
onnx.checker.check_model(onnx_model)
print("✓ ONNX model is valid!")

# Display model information
print("\nModel Information:")
print(f"  IR Version: {onnx_model.ir_version}")
print(f"  Producer: {onnx_model.producer_name} {onnx_model.producer_version}")
print(f"  Opset Version: {onnx_model.opset_import[0].version}")

print("\nModel Inputs:")
for inp in onnx_model.graph.input:
    print(f"  - Name: {inp.name}")

print("\nModel Outputs:")
for out in onnx_model.graph.output:
    print(f"  - Name: {out.name}")

## 10. Test ONNX Inference and Validate

In [ ]:
print("Testing ONNX inference...\n")

# Create ONNX Runtime session
session = ort.InferenceSession(str(ONNX_OUTPUT_PATH))
input_name = session.get_inputs()[0].name

print(f"ONNX Runtime session created")
print(f"  Input name: {input_name}")
print(f"  Output names: {[out.name for out in session.get_outputs()]}")

# Run inference with test input
onnx_outputs = session.run(None, {input_name: test_input})

print(f"\n✓ ONNX inference successful")

## 11. Validate: Compare XGBoost vs ONNX Predictions

In [ ]:
# Get predictions from both models
xgb_labels = model.predict(test_input)
xgb_probs = model.predict_proba(test_input)

onnx_labels = onnx_outputs[0]
onnx_probs = onnx_outputs[1]

# Compare labels
print("Prediction Comparison:")
print("=" * 70)
print(f"{'Sample':<8} {'XGBoost':<15} {'ONNX':<15} {'Match':<10}")
print("=" * 70)

matches = 0
for i in range(len(test_input)):
    xgb_label_name = label_names[xgb_labels[i]]
    onnx_label_name = label_names[onnx_labels[i]]
    match = "✓" if xgb_labels[i] == onnx_labels[i] else "✗"
    if xgb_labels[i] == onnx_labels[i]:
        matches += 1
    
    print(f"{i+1:<8} {xgb_label_name:<15} {onnx_label_name:<15} {match:<10}")

print("=" * 70)
label_match_rate = matches / len(test_input) * 100
print(f"\nLabel Match Rate: {matches}/{len(test_input)} ({label_match_rate:.1f}%)")

# Compare probabilities
prob_diff = np.abs(xgb_probs - onnx_probs)
max_diff = prob_diff.max()
mean_diff = prob_diff.mean()

print(f"\nProbability Comparison:")
print(f"  Max difference:  {max_diff:.8f}")
print(f"  Mean difference: {mean_diff:.8f}")
print(f"  Status: {'✓ Acceptable' if max_diff < 1e-5 else '⚠ Check model'}")

## 12. Benchmark Inference Speed

In [ ]:
# Benchmark with different batch sizes
batch_sizes = [1, 10, 100, 1000]
n_iterations = 100

results = []

print("Benchmarking inference speed...\n")
print(f"{'Batch Size':<12} {'XGBoost (ms)':<15} {'ONNX (ms)':<15} {'Speedup':<10}")
print("=" * 60)

for batch_size in batch_sizes:
    test_batch = np.random.randn(batch_size, n_features).astype(np.float32)
    
    # XGBoost timing
    start = time.time()
    for _ in range(n_iterations):
        _ = model.predict_proba(test_batch)
    xgb_time = (time.time() - start) / n_iterations * 1000  # ms
    
    # ONNX timing
    start = time.time()
    for _ in range(n_iterations):
        _ = session.run(None, {input_name: test_batch})
    onnx_time = (time.time() - start) / n_iterations * 1000  # ms
    
    speedup = xgb_time / onnx_time
    results.append({
        'batch_size': batch_size,
        'xgb_time': xgb_time,
        'onnx_time': onnx_time,
        'speedup': speedup
    })
    
    print(f"{batch_size:<12} {xgb_time:<15.4f} {onnx_time:<15.4f} {speedup:<10.2f}x")

print("=" * 60)
avg_speedup = sum(r['speedup'] for r in results) / len(results)
print(f"\nAverage ONNX speedup: {avg_speedup:.2f}x")

## 13. Final Export Report

In [ ]:
print("\n" + "=" * 70)
print("ONNX MODEL EXPORT REPORT")
print("=" * 70)

print(f"\n📁 OUTPUT FILES")
print(f"  Model file:    {ONNX_OUTPUT_PATH}")
print(f"  File size:     {ONNX_OUTPUT_PATH.stat().st_size / (1024*1024):.2f} MB")

print(f"\n📊 MODEL SPECIFICATION")
print(f"  Input name:    float_input")
print(f"  Input shape:   [batch_size, {n_features}]")
print(f"  Output names:  label, probabilities")
print(f"  Num classes:   3 (DOWN, NEUTRAL, UP)")
print(f"  Num features:  {n_features}")

if METRICS_PATH.exists():
    print(f"\n📈 TRAINING PERFORMANCE")
    print(f"  Train Accuracy: {metrics.get('train_accuracy', 'N/A'):.4f}")
    print(f"  Val Accuracy:   {metrics.get('val_accuracy', 'N/A'):.4f}")
    print(f"  Train F1:       {metrics.get('train_f1_weighted', 'N/A'):.4f}")
    print(f"  Val F1:         {metrics.get('val_f1_weighted', 'N/A'):.4f}")

print(f"\n✅ VALIDATION RESULTS")
print(f"  ONNX model valid:       ✓")
print(f"  Label match rate:       {label_match_rate:.1f}%")
print(f"  Probability max diff:   {max_diff:.8f}")
print(f"  Probability mean diff:  {mean_diff:.8f}")
print(f"  Validation status:      {'✓ PASSED' if max_diff < 1e-5 and label_match_rate == 100.0 else '⚠ CHECK REQUIRED'}")

print(f"\n⚡ PERFORMANCE BENCHMARK")
print(f"  Average speedup:        {avg_speedup:.2f}x")
print(f"  Single inference (1):   {results[0]['onnx_time']:.4f} ms")
print(f"  Batch inference (1000): {results[-1]['onnx_time']:.4f} ms")

print(f"\n📦 FILES FOR GO BOT")
print(f"  1. {ONNX_OUTPUT_PATH.name}")
print(f"  2. {FEATURES_PATH.name}")
print(f"  3. {METRICS_PATH.name}")

print("\n" + "=" * 70)
print("✓ EXPORT COMPLETED SUCCESSFULLY")
print("=" * 70)

## Next Steps for Go Integration

### 1. Install ONNX Runtime Go Bindings

```bash
go get github.com/yalue/onnxruntime_go
```

### 2. Load Model in Go

```go
import "github.com/yalue/onnxruntime_go"

// Load ONNX model
session, err := onnxruntime_go.NewSession("models/xgboost_model.onnx")
if err != nil {
    log.Fatal(err)
}
defer session.Destroy()

// Prepare input tensor (shape: [1, 23])
inputTensor := []float32{
    close, log_ret_1m, log_ret_5m, ema_5, ema_9, ema_21, ema_50,
    rsi_7, rsi_14, bb_upper, bb_middle, bb_lower, bb_width,
    macd, macd_signal, macd_histogram, volume_ratio,
    sentiment_1h, sentiment_24h, mentions_zscore, sentiment_velocity,
    hour_sin, hour_cos,
}

// Run inference
outputs, err := session.Run([][]float32{inputTensor})
if err != nil {
    log.Fatal(err)
}

// Get predictions
label := outputs[0][0]         // Predicted class (0, 1, or 2)
probabilities := outputs[1]     // [p_down, p_neutral, p_up]
```

### 3. Feature Order (CRITICAL)

Ensure features are in the exact same order as `features.json`:

1. close
2. log_ret_1m
3. log_ret_5m
4. ema_5
5. ema_9
6. ema_21
7. ema_50
8. rsi_7
9. rsi_14
10. bb_upper
11. bb_middle
12. bb_lower
13. bb_width
14. macd
15. macd_signal
16. macd_histogram
17. volume_ratio
18. sentiment_1h
19. sentiment_24h
20. mentions_zscore
21. sentiment_velocity
22. hour_sin
23. hour_cos

### 4. Testing Checklist

- [ ] Verify model loads without errors
- [ ] Test inference with sample data
- [ ] Compare predictions with Python results
- [ ] Benchmark latency (should be <1ms for single inference)
- [ ] Test with edge cases (NaN, extreme values)
- [ ] Run backtests with ONNX model
- [ ] Monitor prediction distribution in production

### 5. Production Considerations

- **Model versioning**: Include timestamp in filename (e.g., `xgboost_model_20260207.onnx`)
- **Monitoring**: Log prediction distribution (% DOWN/NEUTRAL/UP)
- **Fallback**: Keep previous model version as backup
- **Retraining**: Schedule periodic retraining (weekly/monthly)
- **A/B testing**: Compare new vs old model predictions